# LABII · Generador de propuestas comerciales técnicas para NubeAndina: Whisper + OKF + Jev

Construimos sobre el RAG de `raglab.ipynb` (Flask + chat + agente ReAct) una **herramienta interna de generación de propuestas comerciales técnicas** para NubeAndina Consulting (consultora AWS ficticia). El equipo comercial la usa después de cada llamada de descubrimiento: la herramienta escucha la llamada, consulta el catálogo y entrega una propuesta con arquitectura y precios, lista para revisar y enviar.

1. **Etapa 1 · Whisper:** la llamada con el cliente (MP3) se transcribe y entra al RAG con marcas de tiempo.
2. **Etapa 2 · OKF:** catálogo de servicios, tarifas y fundamentos de AWS en formato Open Knowledge Format. El agente lo lee por ruta, no por similitud.
3. **Etapa 3 · Jev:** evaluaciones "System 1": decisiones tipadas (choice, score, sí/no) en milisegundos.
4. **Bonus:** el agente dibuja la arquitectura propuesta con iconos AWS y una diapositiva con precios (`gpt-image-2.5-flare`).

Caso: **RitmoFit**, empresa ficticia de fitness conectado, inspirada en la [arquitectura de recomendaciones en tiempo real de Peloton en AWS](https://www.youtube.com/watch?v=ym_Gz_zH7w8).

> El código está en inglés (variables, funciones, comentarios y prompts). Los datos y lo que ve el cliente siguen en español: la llamada, el bundle OKF, las propuestas, el diagrama y la diapositiva.

## Antes de comenzar
1. **Entorno de ejecución → Cambiar tipo → GPU T4** (para Whisper). Sin GPU también funciona, más lento.
2. En **Secrets (🔑)** agrega y habilita "Notebook access":
   - `OPENAI_API_KEY` (obligatoria)
   - `JEV_API_KEY` (obligatoria para la Etapa 3 y el servidor final, formato `jv_live_…`)
3. Ten a mano `llamadanubeandina.mp3` (o `transcripcion_respaldo.json` si no hay audio).

In [ ]:
!pip install -q -U langchain langchain-openai langchain-community langgraph faiss-cpu \
    openai openai-whisper python-frontmatter diagrams

In [ ]:
import os
from google.colab import userdata

def secret(name, required=False):
    try:
        return userdata.get(name)
    except Exception:
        if required:
            raise RuntimeError(f"Missing Colab secret {name}")
        print(f"⚠️ No {name}")
        return None

os.environ["OPENAI_API_KEY"] = secret("OPENAI_API_KEY", required=True)
os.environ["JEV_API_KEY"] = secret("JEV_API_KEY", required=True)

# Chat model: cheap, with tool calling. Change it here if your account doesn't have it.
MODEL = "gpt-6-luna"               # cheapest in the GPT-6 line (USD 0.10 / 0.50 per 1M tokens)
EFFORT = "medium"                  # reasoning: "low" is cheaper and faster, "medium" gives better proposals
EMBEDDINGS_MODEL = "text-embedding-3-small"
IMAGE_MODEL = "gpt-image-2.5-flare"     # bonus: fast and cheap

from langchain_openai import ChatOpenAI

def create_llm():
    # A stuck request fails after 2 min instead of hanging (OpenAI's default is 10 min, retried twice)
    limits = dict(timeout=120, max_retries=1)
    if MODEL.startswith("gpt-4"):      # non-reasoning fallback, e.g. "gpt-4.1-mini"
        return ChatOpenAI(model=MODEL, temperature=0.2, **limits)
    return ChatOpenAI(model=MODEL, reasoning_effort=EFFORT, **limits)    # reasoning models: no temperature

print(create_llm().invoke("Reply only: ready").text)

## Chat (HTML)
Mismo frontend de `raglab.ipynb`, ahora muestra las respuestas en Markdown (las propuestas traen tablas).
- El mensaje de bienvenida trae el botón **📤 Subir consulta**: sube el audio de la llamada (o la transcripción `.json`) a `/upload_audio`, que la transcribe con Whisper y la agrega al índice.
- Un *loader* (tres puntos) aparece mientras el agente responde o mientras se procesa la consulta.

In [ ]:
html = """<!DOCTYPE html>
<html lang="en">

<head>
  <meta charset="UTF-8">
  <title>RAGLabChatbot</title>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <meta http-equiv="X-UA-Compatible" content="ie=edge">
  <link rel="stylesheet" href="{{ url_for('static', filename='styles/name.css') }}">
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=Roboto+Condensed:wght@400;700&display=swap" rel="stylesheet">
    <style>
:root {
  --gradient-start: #2E4590;
  --gradient-mid: #3A1D8A;
  --gradient-end: #2A0059;
  --ai-msg-bg: #3A1D8A;
  --user-msg-bg: #ffffff;
  --input-bg: rgba(30, 0, 65, 0.8);
  --text-dark: #333333;
  --text-light: #ffffff;
  --purple-dark: rgba(58, 29, 138, 1);
  --purple-dark-transparent: rgba(58, 29, 138, 0.25);
  --purple-blue: rgba(46, 69, 144, 1);
}

html {
  box-sizing: border-box;
}

*,
*:before,
*:after {
  margin: 0;
  padding: 0;
  box-sizing: inherit;
}

body {
  display: flex;
  justify-content: center;
  align-items: center;
  height: 100vh;
  background: radial-gradient(ellipse at top left, var(--gradient-start) 0%, var(--gradient-mid) 50%, var(--gradient-end) 100%);
  font-family: 'Roboto Condensed', -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
  overflow: hidden;
}

.msger {
  display: flex;
  flex-flow: column nowrap;
  justify-content: space-between;
  width: 50%;
  height: 100vh;
  background: transparent;
  border: none;
  box-shadow: none;
  margin: 0;
  padding: 0;
}

.msger-header {
  display: flex;
  justify-content: center;
  align-items: center;
  padding: 20px 20px 10px 20px;
  background: transparent;
  border: none;
}

.header-left {
  display: none;
}

.menu-button {
  display: none;
}

.score-circle {
  display: none;
}

.msger-header-title {
  font-size: 24px;
  font-weight: 600;
  color: var(--text-light);
  text-align: center;
  display: flex;
  align-items: center;
  gap: 15px;
  font-family: 'Roboto Condensed', sans-serif;
  font-weight: 700;
  letter-spacing: 1px;
}

.aural-avatar {
  position: relative;
  width: 60px;
  height: 60px;
  border-radius: 50%;
  background: #000;
  overflow: hidden;
  display: flex;
  align-items: center;
  justify-content: center;
  box-shadow: 0 0 20px rgba(58, 29, 138, 0.5), inset 0 0 30px rgba(46, 69, 144, 0.3);
}

.aural-avatar::before {
  content: '';
  position: absolute;
  width: 100%;
  height: 100%;
  background: radial-gradient(circle at 30% 30%, rgba(58, 29, 138, 0.8) 0%, rgba(46, 69, 144, 0.4) 30%, transparent 70%);
  animation: pulse 3s ease-in-out infinite;
}

.aural-avatar::after {
  content: '';
  position: absolute;
  width: 120%;
  height: 120%;
  background: conic-gradient(from 0deg, transparent, rgba(58, 29, 138, 0.3), transparent, rgba(46, 69, 144, 0.3), transparent);
  animation: rotate 8s linear infinite;
  border-radius: 50%;
}

@keyframes pulse {
  0%, 100% {
    opacity: 0.6;
    transform: scale(1);
  }
  50% {
    opacity: 1;
    transform: scale(1.1);
  }
}

@keyframes rotate {
  0% {
    transform: rotate(0deg);
  }
  100% {
    transform: rotate(360deg);
  }
}

.particle {
  position: absolute;
  width: 3px;
  height: 3px;
  background: rgba(255, 255, 255, 0.8);
  border-radius: 50%;
  box-shadow: 0 0 6px rgba(58, 29, 138, 1), 0 0 12px rgba(46, 69, 144, 0.8);
  animation: float 4s ease-in-out infinite;
}

.particle:nth-child(1) {
  top: 20%;
  left: 30%;
  animation-delay: 0s;
  animation-duration: 3s;
}

.particle:nth-child(2) {
  top: 50%;
  left: 60%;
  animation-delay: 0.5s;
  animation-duration: 4s;
}

.particle:nth-child(3) {
  top: 70%;
  left: 25%;
  animation-delay: 1s;
  animation-duration: 3.5s;
}

.particle:nth-child(4) {
  top: 30%;
  left: 70%;
  animation-delay: 1.5s;
  animation-duration: 4.5s;
}

.particle:nth-child(5) {
  top: 60%;
  left: 45%;
  animation-delay: 2s;
  animation-duration: 3s;
}

@keyframes float {
  0%, 100% {
    transform: translate(0, 0) scale(1);
    opacity: 0.6;
  }
  25% {
    transform: translate(10px, -10px) scale(1.2);
    opacity: 1;
  }
  50% {
    transform: translate(-5px, 5px) scale(0.8);
    opacity: 0.8;
  }
  75% {
    transform: translate(-10px, -5px) scale(1.1);
    opacity: 0.9;
  }
}

.light-trail {
  position: absolute;
  width: 2px;
  height: 40px;
  background: linear-gradient(to bottom, transparent, rgba(58, 29, 138, 0.8), transparent);
  border-radius: 50%;
  top: 50%;
  left: 50%;
  transform-origin: center;
  animation: trail-rotate 6s linear infinite;
  box-shadow: 0 0 10px rgba(58, 29, 138, 0.6);
}

.light-trail:nth-child(6) {
  animation-delay: 0s;
  transform: translate(-50%, -50%) rotate(0deg) translateY(-25px);
}

.light-trail:nth-child(7) {
  animation-delay: 2s;
  transform: translate(-50%, -50%) rotate(120deg) translateY(-25px);
}

.light-trail:nth-child(8) {
  animation-delay: 4s;
  transform: translate(-50%, -50%) rotate(240deg) translateY(-25px);
}

@keyframes trail-rotate {
  0% {
    opacity: 0.3;
    transform: translate(-50%, -50%) rotate(0deg) translateY(-25px);
  }
  50% {
    opacity: 1;
  }
  100% {
    opacity: 0.3;
    transform: translate(-50%, -50%) rotate(360deg) translateY(-25px);
  }
}

.chat-glass-container {
  flex: 1;
  display: flex;               /* lets .msger-chat take the remaining height and scroll */
  flex-direction: column;
  margin: 0 20px 20px 20px;
  border-radius: 20px;
  backdrop-filter: blur(30px) saturate(180%);
  -webkit-backdrop-filter: blur(30px) saturate(180%);
  background: rgba(58, 29, 138, 0.15);
  border: 1px solid rgba(255, 255, 255, 0.2);
  box-shadow: 0 8px 32px 0 rgba(20, 15, 80, 0.4),
              inset 0 1px 0 rgba(255, 255, 255, 0.2),
              inset 0 -1px 0 rgba(255, 255, 255, 0.1);
  overflow: hidden;
  position: relative;
  min-height: 0;
}

.chat-glass-container::before {
  content: '';
  position: absolute;
  top: 0;
  left: 0;
  right: 0;
  height: 1px;
  background: linear-gradient(90deg,
    transparent,
    rgba(255, 255, 255, 0.5) 50%,
    transparent
  );
  pointer-events: none;
  z-index: 1;
}

.chat-glass-container::after {
  content: '';
  position: absolute;
  top: 0;
  left: 0;
  right: 0;
  bottom: 0;
  background: radial-gradient(circle at 20% 20%, rgba(255, 255, 255, 0.1) 0%, transparent 50%);
  pointer-events: none;
  z-index: 0;
}

.msger-chat {
  flex: 1;
  overflow-y: auto;
  padding: 20px;
  background: transparent;
  display: flex;
  flex-direction: column;
  gap: 15px;
  min-height: 0;
  position: relative;
  z-index: 1;
}

.msger-chat::-webkit-scrollbar {
  width: 8px;
}

.msger-chat::-webkit-scrollbar-track {
  background: rgba(255, 255, 255, 0.05);
  border-radius: 4px;
  margin: 10px 0;
}

.msger-chat::-webkit-scrollbar-thumb {
  background: rgba(255, 255, 255, 0.3);
  border-radius: 4px;
  border: 1px solid rgba(255, 255, 255, 0.1);
  backdrop-filter: blur(10px);
  -webkit-backdrop-filter: blur(10px);
}

.msger-chat::-webkit-scrollbar-thumb:hover {
  background: rgba(255, 255, 255, 0.4);
}

.msg {
  display: flex;
  align-items: flex-end;
  margin-bottom: 0;
}

.msg-img {
  display: none;
}

.msg-bubble {
  max-width: 70%;
  padding: 15px 20px;
  border-radius: 25px;
  background: var(--ai-msg-bg);
  color: var(--text-light);
  word-wrap: break-word;
  position: relative;
  backdrop-filter: blur(20px) saturate(180%);
  -webkit-backdrop-filter: blur(20px) saturate(180%);
  background: rgba(58, 29, 138, 0.25);
  border: 1px solid rgba(255, 255, 255, 0.18);
  box-shadow: 0 8px 32px 0 rgba(20, 15, 80, 0.37),
              inset 0 1px 0 rgba(255, 255, 255, 0.3);
  overflow: hidden;
}

.msg-bubble::before {
  content: '';
  position: absolute;
  top: 0;
  left: 0;
  right: 0;
  height: 1px;
  background: linear-gradient(90deg,
    transparent,
    rgba(255, 255, 255, 0.4) 50%,
    transparent
  );
  pointer-events: none;
}

.msg-info {
  display: none;
}

.msg-text {
  font-size: 16px;
  line-height: 1.5;
  position: relative;
  z-index: 1;
}

.left-msg {
  justify-content: flex-start;
}

.left-msg .msg-bubble {
  background: rgba(58, 29, 138, 0.25);
  color: var(--text-light);
  border-radius: 25px;
  backdrop-filter: blur(20px) saturate(180%);
  -webkit-backdrop-filter: blur(20px) saturate(180%);
  border: 1px solid rgba(255, 255, 255, 0.18);
  box-shadow: 0 8px 32px 0 rgba(20, 15, 80, 0.37),
              inset 0 1px 0 rgba(255, 255, 255, 0.3);
}

.right-msg {
  flex-direction: row;
  justify-content: flex-end;
}

.right-msg .msg-bubble {
  background: rgba(255, 255, 255, 0.9);
  color: var(--text-dark);
  border-radius: 25px;
  backdrop-filter: blur(20px) saturate(180%);
  -webkit-backdrop-filter: blur(20px) saturate(180%);
  border: 1px solid rgba(58, 29, 138, 0.4);
  box-shadow: 0 8px 32px 0 rgba(58, 29, 138, 0.3),
              inset 0 1px 0 rgba(255, 255, 255, 0.8),
              0 0 0 1px rgba(58, 29, 138, 0.2);
}

.msger-inputarea {
  display: flex;
  padding: 20px;
  background: transparent;
  border: none;
  gap: 10px;
}

.msger-input {
  flex: 1;
  background: var(--input-bg);
  border: none;
  border-radius: 25px;
  padding: 15px 20px;
  font-size: 16px;
  color: rgba(255, 255, 255, 0.9);
  outline: none;
}

.msger-input::placeholder {
  color: rgba(255, 255, 255, 0.5);
}

.msger-send-btn {
  display: none;
}

/* Proposals come back as Markdown with tables */
.msg-text table {
  display: block;
  overflow-x: auto;
  border-collapse: collapse;
  margin: 10px 0;
}

.msg-text th,
.msg-text td {
  border: 1px solid rgba(255, 255, 255, 0.25);
  padding: 4px 8px;
}

.msg-text ul,
.msg-text ol {
  padding-left: 20px;
}

.upload-form {
  display: flex;
  flex-wrap: wrap;
  gap: 8px;
  margin-top: 12px;
}

.upload-client {
  flex: 1;
  min-width: 120px;
  background: var(--input-bg);
  border: none;
  border-radius: 20px;
  padding: 10px 15px;
  font-size: 14px;
  color: var(--text-light);
  outline: none;
}

.upload-btn {
  background: var(--text-light);
  color: var(--purple-dark);
  border: none;
  border-radius: 20px;
  padding: 10px 18px;
  font-family: inherit;
  font-size: 14px;
  font-weight: 700;
  cursor: pointer;
}

.upload-btn:disabled {
  opacity: 0.5;
  cursor: wait;
}

/* Loader: three bouncing dots while the server works */
.typing {
  display: inline-flex;
  gap: 4px;
  margin-right: 10px;
  vertical-align: middle;
}

.typing span {
  width: 8px;
  height: 8px;
  border-radius: 50%;
  background: rgba(255, 255, 255, 0.8);
  animation: typing-bounce 1.2s ease-in-out infinite;
}

.typing span:nth-child(2) {
  animation-delay: 0.15s;
}

.typing span:nth-child(3) {
  animation-delay: 0.3s;
}

@keyframes typing-bounce {
  0%, 60%, 100% {
    transform: translateY(0);
    opacity: 0.4;
  }
  30% {
    transform: translateY(-5px);
    opacity: 1;
  }
}

.loader-label {
  opacity: 0.8;
  font-style: italic;
}

/* Jev result under each answer */
.checks {
  margin-top: 10px;
  padding-top: 8px;
  font-size: 13px;
  opacity: 0.85;
  border-top: 1px solid rgba(255, 255, 255, 0.2);
}

/* Diagram and slide */
.msg-text img {
  display: block;
  max-width: 100%;
  margin: 8px 0 12px;
  border-radius: 12px;
  background: #ffffff;
}
</style>
  <script src="https://ajax.googleapis.com/ajax/libs/jquery/3.2.1/jquery.min.js"></script>
</head>

<body>
  <!-- partial:index.partial.html -->
  <section class="msger">
    <header class="msger-header">
      <div class="msger-header-title">
        <div class="aural-avatar">
          <div class="particle"></div>
          <div class="particle"></div>
          <div class="particle"></div>
          <div class="particle"></div>
          <div class="particle"></div>
          <div class="light-trail"></div>
          <div class="light-trail"></div>
          <div class="light-trail"></div>
        </div>
        NubeAndina
      </div>
    </header>

    <div class="chat-glass-container">
      <main class="msger-chat">
      <div class="msg left-msg">
        <div class="msg-img" style="background-image: url(https://image.flaticon.com/icons/svg/327/327779.svg)"></div>

        <div class="msg-bubble">
          <div class="msg-info">
            <div class="msg-info-name">Fukuro</div>
            <div class="msg-info-time">12:45</div>
          </div>

          <div class="msg-text">
            Generador de propuestas de NubeAndina. Empieza subiendo la consulta (audio de la llamada o transcripción .json); luego indica el cliente y la necesidad, y armo la propuesta técnica y comercial.
            <form class="upload-form" id="uploadForm">
              <input type="text" class="upload-client" id="uploadClient" placeholder="Cliente (p. ej. ritmofit)" autocomplete="off">
              <input type="file" id="uploadFile" accept="audio/*,.json" hidden>
              <button type="submit" class="upload-btn" id="uploadBtn">📤 Subir consulta</button>
            </form>
          </div>
        </div>
      </div>

    </main>
    </div>

    <form class="msger-inputarea">
      <input type="text" autocomplete="off" class="msger-input" id="textInput" placeholder="Message">
      <button type="submit" class="msger-send-btn" style="display: none;">Send</button>
    </form>
  </section>
  <!-- partial -->
  <script src='https://use.fontawesome.com/releases/v5.0.13/js/all.js'></script>
  <script src="https://cdn.jsdelivr.net/npm/marked/marked.min.js"></script>
  <script>

    const msgerForm = get(".msger-inputarea");
    const msgerInput = get(".msger-input");
    const msgerChat = get(".msger-chat");


    // Icons made by Freepik from www.flaticon.com
    const BOT_IMG = "https://image.flaticon.com/icons/svg/327/327779.svg";
    const PERSON_IMG = "https://image.flaticon.com/icons/svg/145/145867.svg";
    const BOT_NAME = "    ChatBot";
    const PERSON_NAME = "You";

    msgerForm.addEventListener("submit", event => {
      event.preventDefault();

      const msgText = msgerInput.value;
      if (!msgText) return;

      appendMessage(PERSON_NAME, PERSON_IMG, "right", msgText);
      msgerInput.value = "";
      botResponse(msgText);
    });

    function appendMessage(name, img, side, text) {
      //   Simple solution for small apps
      const msgHTML = `
<div class="msg ${side}-msg">
  <div class="msg-img" style="background-image: url(${img})"></div>

  <div class="msg-bubble">
    <div class="msg-info">
      <div class="msg-info-name">${name}</div>
      <div class="msg-info-time">${formatDate(new Date())}</div>
    </div>

    <div class="msg-text">${text}</div>
  </div>
</div>
`;

      msgerChat.insertAdjacentHTML("beforeend", msgHTML);
      scrollToBottom();
    }

    // Shows a "typing" bubble; returns it so the caller can remove it
    function showLoader(label) {
      msgerChat.insertAdjacentHTML("beforeend", `
<div class="msg left-msg">
  <div class="msg-bubble">
    <div class="msg-text"><span class="typing"><span></span><span></span><span></span></span><span class="loader-label">${label}</span></div>
  </div>
</div>`);
      scrollToBottom();
      return msgerChat.lastElementChild;
    }

    function scrollToBottom() {
      msgerChat.scrollTop = msgerChat.scrollHeight;
    }

    session_id = Math.random().toString(36).substring(2, 6)

    function botResponse(rawText) {
      const loader = showLoader("Consultando la llamada y el catálogo…");
      msgerInput.disabled = true;          // one question at a time per session
      $.post("/ask_bot", { msg: rawText, session_id: session_id })
        .done(data => {
          appendMessage(BOT_NAME, BOT_IMG, "left", marked.parse(data.msg) + checksBadge(data.checks));
          if (data.visuals) requestVisuals(data.msg);
        })
        .fail(() => appendMessage(BOT_NAME, BOT_IMG, "left",
          "⚠️ Error del servidor. Revisa la salida de la celda del servidor en Colab."))
        .always(() => {
          loader.remove();
          msgerInput.disabled = false;
          msgerInput.focus();
        });
    }

    // Jev result line; checks is undefined on servers without Jev, null if Jev failed
    function checksBadge(c) {
      if (c === undefined) return "";
      if (c === null) return `<div class="checks">🛡️ Sin revisión de Jev</div>`;
      const mark = v => (v >= 0.5 ? "✔" : "✖");
      return `<div class="checks">🛡️ Jev · tipo: ${c.type} · precios ${mark(c.prices_ok)} ${c.prices_ok}` +
             ` · restricciones ${mark(c.constraints_ok)} ${c.constraints_ok} · cobertura ${c.coverage}/3</div>`;
    }

    function imageTag(src, alt) {
      return `<a href="${src}" target="_blank"><img src="${src}" alt="${alt}"></a>`;
    }

    // Proposals: diagram with AWS icons + slide from the image model
    function requestVisuals(proposal) {
      const loader = showLoader("Dibujando la arquitectura y generando la diapositiva…");
      const usd = n => Math.round(n).toLocaleString("en-US");
      $.ajax({ url: "/visuals", type: "POST", contentType: "application/json", data: JSON.stringify({ proposal }) })
        .done(v => appendMessage(BOT_NAME, BOT_IMG, "left",
          `<b>Arquitectura propuesta</b>${imageTag(v.diagram, "Diagrama de arquitectura")}` +
          `<b>Diapositiva</b> · consultoría USD ${usd(v.totals.consulting)} · AWS USD ${usd(v.totals.aws_monthly)} / mes` +
          imageTag(v.slide, "Diapositiva de la propuesta")))
        .fail(() => appendMessage(BOT_NAME, BOT_IMG, "left",
          "⚠️ No pude generar las imágenes. Revisa la salida de la celda del servidor en Colab."))
        .always(() => loader.remove());
    }

    // Upload consultation: the button opens the file picker, picking a file uploads it
    const uploadForm = get("#uploadForm");
    const uploadFile = get("#uploadFile");
    const uploadClient = get("#uploadClient");
    const uploadBtn = get("#uploadBtn");

    uploadForm.addEventListener("submit", event => {
      event.preventDefault();
      uploadFile.click();
    });

    uploadFile.addEventListener("change", () => {
      const file = uploadFile.files[0];
      if (!file) return;
      const client = uploadClient.value.trim() || "cliente";
      const data = new FormData();
      data.append("audio", file);
      data.append("client", client);

      appendMessage(PERSON_NAME, PERSON_IMG, "right", `📎 ${file.name} · ${client}`);
      uploadBtn.disabled = true;
      const label = file.name.endsWith(".json") ? "Indexando la transcripción…" : "Transcribiendo la llamada con Whisper…";
      const loader = showLoader(label);

      fetch("/upload_audio", { method: "POST", body: data })
        .then(r => r.ok ? r.json() : Promise.reject(r.status))
        .then(res => appendMessage(BOT_NAME, BOT_IMG, "left",
          `✅ Consulta de <b>${res.client}</b> procesada: ${res.minutes} min, ${res.chunks} fragmentos indexados.<br>` +
          `Ahora pide la propuesta, por ejemplo: <i>Genera la propuesta para ${res.client}</i>.`))
        .catch(() => appendMessage(BOT_NAME, BOT_IMG, "left",
          "⚠️ No pude procesar el archivo. Revisa la salida de la celda del servidor en Colab."))
        .finally(() => {
          loader.remove();
          uploadBtn.disabled = false;
          uploadFile.value = "";
        });
    });


    // Utils
    function get(selector, root = document) {
      return root.querySelector(selector);
    }

    function formatDate(date) {
      const h = "0" + date.getHours();
      const m = "0" + date.getMinutes();

      return `${h.slice(-2)}:${m.slice(-2)}`;
    }



  </script>

</body>

</html>"""

## Etapa 1 · Whisper: la llamada del cliente entra al RAG

**Objetivo:** convertir la llamada en fragmentos con marca de tiempo, buscables por el agente.

- `whispr.ipynb` imprimía `result["text"]` y descartaba `result["segments"]`. Los segmentos traen `start`/`end`: eso permite citar "minuto 2:14".
- Una transcripción es texto no estructurado → va al lado **vectorial** (FAISS).
- `transcribe()` devuelve la lista `segments`; si subes `transcripcion_respaldo.json`, se carga tal cual.

In [ ]:
from google.colab import files

uploaded = files.upload()        # upload llamadanubeandina.mp3 (or transcripcion_respaldo.json)
FILE = list(uploaded.keys())[0]
print(FILE)

In [ ]:
import json, torch, whisper

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
whisper_model = None

def transcribe(path):
    """Returns [{start, end, text}] using local Whisper."""
    global whisper_model
    if whisper_model is None:
        whisper_model = whisper.load_model("turbo" if DEVICE == "cuda" else "base", device=DEVICE)
    r = whisper_model.transcribe(path, language="es", fp16=(DEVICE == "cuda"))   # the call is in Spanish
    return [{"start": s["start"], "end": s["end"], "text": s["text"].strip()} for s in r["segments"]]

if FILE.endswith(".json"):                        # fallback without audio
    segments = json.load(open(FILE))
else:
    segments = transcribe(FILE)
    json.dump(segments, open("transcript.json", "w"), ensure_ascii=False, indent=1)

print(DEVICE, len(segments), "segments")
for s in segments[:5]:
    print(f"[{s['start']:6.1f}s] {s['text']}")

**Fragmentar por tiempo, no por caracteres.** `segments_to_docs` agrupa los segmentos en ventanas de ~60 s (`window`) y guarda el cliente y los tiempos en los metadatos (`client`, `start`, `end`). Compáralo con `RecursiveCharacterTextSplitter` del Paso 3 de `raglab.ipynb`.

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

def segments_to_docs(segments, client, window=60):
    docs, buf, start = [], [], None
    def close(end):
        docs.append(Document(" ".join(buf), metadata={
            "source": "transcript", "client": client, "start": round(start), "end": round(end)}))
    for s in segments:
        start = s["start"] if start is None else start
        buf.append(s["text"])
        if s["end"] - start >= window:
            close(s["end"]); buf, start = [], None
    if buf:
        close(segments[-1]["end"])
    return docs

transcript_docs = segments_to_docs(segments, "ritmofit")
transcript_db = FAISS.from_documents(transcript_docs, OpenAIEmbeddings(model=EMBEDDINGS_MODEL))
print(len(transcript_docs), "chunks indexed")

In [ ]:
from langchain_core.tools import tool

@tool
def search_transcripts(query: str) -> str:
    """Searches client calls (in Spanish): needs, budget, timelines, team and constraints."""
    hits = transcript_db.similarity_search(query, k=4)
    return "\n\n".join(
        f"[{d.metadata['client']} {d.metadata['start']}s-{d.metadata['end']}s] {d.page_content}"
        for d in hits)

# ✅ Checkpoint (the call is in Spanish, so Spanish queries match best)
print(search_transcripts.invoke("presupuesto"))

## Etapa 2 · OKF: el conocimiento curado de la empresa

**Objetivo:** que los datos que deben ser exactos (precios, servicios) se lean **por ruta**, no por similitud.

[Open Knowledge Format](https://heise.de/-11332310) (Google, v0.1, junio 2026): una carpeta de "conceptos" en Markdown.
- Frontmatter YAML con `type` obligatorio; recomendados `title`, `description`, `resource`, `tags`, `timestamp`.
- `index.md` en cada carpeta (divulgación progresiva) y `log.md` (historial).
- Los conceptos se enlazan con links Markdown; la ruta es el ID.
- v0.2 agrega `status`, `stale_after`, `generated`, `verified`.

Arquitectura híbrida ([OKF + RAG](https://medium.com/@ravishkhullar/okf-rag-the-ultimate-ai-agent-architecture-26b9ceed44f1)): OKF para lo curado, RAG vectorial para lo no estructurado (la llamada).

### Cargar el bundle OKF
Sube `okf.zip` (24 archivos, en la carpeta LABII). Si no lo tienes a mano, ejecuta la celda **Plan B**, que crea el mismo bundle desde el diccionario `OKF_FILES`. El contenido del bundle sigue en español: es la base de conocimiento de una consultora latinoamericana.

In [ ]:
import pathlib, zipfile
from google.colab import files

OKF = pathlib.Path("okf")
if not OKF.exists():
    uploaded = files.upload()                    # select okf.zip
    zipfile.ZipFile(next(iter(uploaded))).extractall(".")
print(len(list(OKF.rglob("*.md"))), "OKF files")
!find okf -name '*.md' | sort

In [ ]:
# Plan B: builds the bundle from the notebook (not needed if you already uploaded okf.zip)
# The OKF content stays in Spanish: it is the company's knowledge base and must match okf.zip
import pathlib

OKF = pathlib.Path("okf")

OKF_FILES = {
'index.md': """---
type: Index
title: Base de conocimiento de NubeAndina Consulting
description: Punto de entrada para el agente. Empieza aquí y sigue los enlaces.
timestamp: 2026-10-05T00:00:00Z
---
# NubeAndina Consulting: base de conocimiento

- [Empresa: quiénes somos, servicios y tarifas](company/index.md)
- [AWS: fundamentos de servicios, costos de referencia y patrones](aws/index.md)
""",

'log.md': """---
type: Log
---
# Cambios

## 2026-10-05
- Se agregan los servicios de plataforma de recomendaciones en tiempo real y pipeline MLOps.
- Se agrega el patrón de recomendaciones en tiempo real.
""",

'company/index.md': """---
type: Index
title: Empresa
description: Quiénes somos, catálogo de servicios y tarifas oficiales.
---
# Empresa

- [Quiénes somos y condiciones comerciales](about.md)
- [Tarifas 2026: ÚNICA fuente válida de precios de consultoría](rate-card.md)
- [Catálogo de servicios](services/index.md)
""",

'company/about.md': """---
type: Company
title: NubeAndina Consulting
description: Consultora AWS ficticia para el laboratorio. Oficinas en Bogotá y Ciudad de México.
tags: [empresa, condiciones]
timestamp: 2026-10-01T00:00:00Z
---
# NubeAndina Consulting

Consultora ficticia especializada en arquitectura, datos y ML sobre AWS. Equipo en Bogotá y Ciudad de México.

## Cómo trabajamos
1. Assessment: entendemos la necesidad y el estado actual.
2. Diseño e implementación por fases, con entregas cada 2 semanas.
3. Operación gestionada opcional.

## Condiciones comerciales
- Precios en USD, sin impuestos.
- 50 % al inicio y 50 % a la entrega.
- Vigencia de la propuesta: 30 días.
- Los costos de AWS los paga el cliente directamente a AWS; no están incluidos en nuestras tarifas.
- Solo ofrecemos servicios sobre AWS. No ofrecemos servicios sobre Azure ni Google Cloud.
""",

'company/rate-card.md': """---
type: RateCard
title: Tarifas NubeAndina Consulting 2026
description: Precios oficiales de consultoría AWS. Única fuente válida de precios.
tags: [precios, consultoria]
timestamp: 2026-10-01T00:00:00Z
status: stable
stale_after: 2027-03-31
---
# Tarifas 2026 (USD)

| Servicio | Unidad | Precio |
| --- | --- | --- |
| [Assessment de arquitectura](services/assessment.md) | paquete 1 semana | 1,800 |
| [Plataforma de recomendaciones en tiempo real](services/realtime-recs-platform.md) | paquete MVP | 9,500 |
| [Pipeline MLOps: entrenamiento diario, orquestación y alertas](services/mlops-pipeline.md) | paquete | 7,000 |
| [Migración lift-and-shift](services/migration.md) | por workload | 4,500 |
| [Well-Architected Review](services/well-architected-review.md) | paquete | 2,200 |
| [Operación gestionada](services/managed-ops.md) | mensual | 900 |
| Arquitecto cloud senior | hora | 95 |

Cualquier servicio que no esté en esta tabla no se ofrece ni se cotiza.
""",

'company/services/index.md': """---
type: Index
title: Catálogo de servicios
description: Servicios de consultoría que ofrecemos. Precios en ../rate-card.md.
---
# Catálogo de servicios

- [Assessment de arquitectura](assessment.md): punto de partida de todo proyecto.
- [Plataforma de recomendaciones en tiempo real](realtime-recs-platform.md)
- [Pipeline MLOps](mlops-pipeline.md)
- [Migración lift-and-shift](migration.md)
- [Well-Architected Review](well-architected-review.md)
- [Operación gestionada](managed-ops.md)

Precios: [tarifas](../rate-card.md).
""",

'company/services/assessment.md': """---
type: Service
title: Assessment de arquitectura
description: Diagnóstico de 1 semana del estado actual y la arquitectura objetivo.
tags: [assessment, descubrimiento]
---
# Assessment de arquitectura

- Duración: 1 semana.
- Alcance: entrevistas, revisión de datos y sistemas actuales, requisitos no funcionales (latencia, costo, equipo).
- Entregables: arquitectura objetivo, plan por fases, estimación de costos AWS.
- Requisito previo para la [plataforma de recomendaciones](realtime-recs-platform.md) y el [pipeline MLOps](mlops-pipeline.md).
- Precio: ver [tarifas](../rate-card.md).
""",

'company/services/realtime-recs-platform.md': """---
type: Service
title: Plataforma de recomendaciones en tiempo real
description: MVP de inferencia en línea con contexto, sobre servicios administrados de AWS.
tags: [ml, recomendaciones, tiempo-real]
---
# Plataforma de recomendaciones en tiempo real (paquete MVP)

- Duración: 6 a 8 semanas.
- Alcance:
  - Feature store de baja latencia en DynamoDB.
  - Grafo miembro-clase-instructor en Neptune para generar candidatos.
  - Endpoint de inferencia en SageMaker (sin Kubernetes).
  - API de recomendaciones (API Gateway + Lambda) con objetivo de latencia < 200 ms.
  - Registro de eventos (MSK Serverless o Kinesis) hacia S3 para reentrenar.
- No incluye: el pipeline de entrenamiento y orquestación (ver [Pipeline MLOps](mlops-pipeline.md)).
- Patrón de referencia: [recomendaciones en tiempo real](../../aws/patterns/recomendaciones-tiempo-real.md).
- Precio: ver [tarifas](../rate-card.md).
""",

'company/services/mlops-pipeline.md': """---
type: Service
title: Pipeline MLOps
description: Entrenamiento diario automatizado, registro de modelos, orquestación y alertas.
tags: [ml, mlops, orquestacion]
---
# Pipeline MLOps

- Duración: 3 a 4 semanas (puede ir en paralelo con la plataforma).
- Alcance: preprocesamiento en EMR (Spark), entrenamiento en SageMaker, registro de modelos,
  orquestación con Amazon MWAA (Airflow) y alertas de fallo con CloudWatch + SNS.
- Precio: ver [tarifas](../rate-card.md).
""",

'company/services/migration.md': """---
type: Service
title: Migración lift-and-shift
description: Mover un workload existente a AWS sin rediseñarlo.
tags: [migracion]
---
# Migración lift-and-shift

- Duración: 2 a 3 semanas por workload.
- Alcance: inventario, migración a EC2/RDS, pruebas y corte.
- Precio: por workload, ver [tarifas](../rate-card.md).
""",

'company/services/managed-ops.md': """---
type: Service
title: Operación gestionada
description: Operación mensual de la plataforma después del go-live.
tags: [operacion, soporte]
---
# Operación gestionada

- Monitoreo 8x5, respuesta a incidentes, parches y reporte mensual de costos.
- Precio: mensual, ver [tarifas](../rate-card.md).
""",

'company/services/well-architected-review.md': """---
type: Service
title: Well-Architected Review
description: Revisión de una carga existente contra el AWS Well-Architected Framework.
tags: [revision, buenas-practicas]
---
# Well-Architected Review

- Duración: 1 a 2 semanas.
- Alcance: revisión de los seis pilares del AWS Well-Architected Framework (excelencia operacional,
  seguridad, confiabilidad, eficiencia de rendimiento, optimización de costos y sostenibilidad).
- Entregables: informe de riesgos priorizados y plan de remediación.
- Precio: ver [tarifas](../rate-card.md).
""",

'aws/index.md': """---
type: Index
title: AWS
description: Fundamentos de servicios, costos de referencia y patrones de arquitectura.
---
# AWS

- [Fundamentos de servicios](basics/index.md)
- [Costos de referencia (ilustrativos)](basics/pricing-models.md)
- [Patrón: recomendaciones en tiempo real](patterns/recomendaciones-tiempo-real.md)
""",

'aws/basics/index.md': """---
type: Index
title: Fundamentos de servicios AWS
description: Una ficha por servicio usado en nuestras propuestas.
---
# Fundamentos

- [Amazon S3](s3.md): almacenamiento de objetos, data lake.
- [Amazon EMR](emr.md): Spark administrado para preprocesar datos.
- [Amazon DynamoDB](dynamodb.md): clave-valor de baja latencia, feature store.
- [Amazon Neptune](neptune.md): base de datos de grafos.
- [Amazon MSK](msk.md): Kafka administrado para eventos.
- [Amazon SageMaker](sagemaker.md): entrenamiento y endpoints de inferencia.
- [Amazon EKS](eks.md): Kubernetes administrado.
- [Amazon MWAA](mwaa.md): Airflow administrado.
- [Costos de referencia](pricing-models.md)
""",

'aws/basics/s3.md': """---
type: Concept
title: Amazon S3
tags: [almacenamiento, data-lake]
resource: https://aws.amazon.com/s3/
---
# Amazon S3
Almacenamiento de objetos. Se usa como data lake del historial y destino de eventos para reentrenar.
Cobro por GB almacenado al mes y por solicitudes.
""",

'aws/basics/emr.md': """---
type: Concept
title: Amazon EMR
tags: [spark, batch]
resource: https://aws.amazon.com/emr/
---
# Amazon EMR
Clústeres Spark administrados (o EMR Serverless). Ideal para preprocesar el historial diario y poblar
el feature store y el grafo. Cobro por hora de cómputo; los jobs diarios cortos mantienen el costo bajo.
""",

'aws/basics/dynamodb.md': """---
type: Concept
title: Amazon DynamoDB
tags: [nosql, baja-latencia, feature-store]
resource: https://aws.amazon.com/dynamodb/
---
# Amazon DynamoDB
Base clave-valor totalmente administrada, latencia de milisegundos de un dígito.
Patrón común como feature store en línea. Modo on-demand: se paga por lectura/escritura.
""",

'aws/basics/neptune.md': """---
type: Concept
title: Amazon Neptune
tags: [grafo]
resource: https://aws.amazon.com/neptune/
---
# Amazon Neptune
Base de datos de grafos administrada. Modela miembros, clases e instructores como nodos y las clases
tomadas como aristas, para generar candidatos de recomendación. Existe en modo serverless.
""",

'aws/basics/msk.md': """---
type: Concept
title: Amazon MSK
tags: [streaming, eventos, kafka]
resource: https://aws.amazon.com/msk/
---
# Amazon MSK
Apache Kafka administrado. MSK Serverless evita dimensionar brokers. Alternativa más simple para
equipos pequeños: Amazon Kinesis Data Streams.
""",

'aws/basics/sagemaker.md': """---
type: Concept
title: Amazon SageMaker
tags: [ml, inferencia, entrenamiento]
resource: https://aws.amazon.com/sagemaker/
---
# Amazon SageMaker
Entrenamiento administrado, registro de modelos y endpoints de inferencia en tiempo real con autoescalado.
Recomendado para equipos sin experiencia en Kubernetes: no hay clústeres que operar.
""",

'aws/basics/eks.md': """---
type: Concept
title: Amazon EKS
tags: [kubernetes, contenedores]
resource: https://aws.amazon.com/eks/
---
# Amazon EKS
Kubernetes administrado. Muy flexible (por ejemplo, GPUs para entrenamiento), pero requiere un equipo con
experiencia en Kubernetes para operarlo. Si el cliente no la tiene, preferir SageMaker.
""",

'aws/basics/mwaa.md': """---
type: Concept
title: Amazon MWAA
tags: [orquestacion, airflow]
resource: https://aws.amazon.com/managed-workflows-for-apache-airflow/
---
# Amazon MWAA
Apache Airflow administrado para orquestar el pipeline diario (preprocesar, entrenar, registrar, desplegar),
con reintentos y alertas.
""",

'aws/basics/pricing-models.md': """---
type: Concept
title: Costos AWS de referencia para un MVP de recomendaciones
description: Valores ILUSTRATIVOS para el laboratorio. No son una cotización de AWS.
tags: [precios-aws, estimacion]
timestamp: 2026-10-05T00:00:00Z
status: draft
---
# Costos AWS de referencia (ilustrativos)

Escenario: ~200.000 miembros, inferencia en línea < 200 ms, reentrenamiento diario.
Valores aproximados en USD por mes, solo para el laboratorio. Validar siempre con AWS Pricing Calculator.

| Servicio | Uso supuesto | USD/mes (rango) |
| --- | --- | --- |
| S3 | 4 TB de historial + eventos | 90 – 120 |
| DynamoDB | feature store on-demand | 300 – 600 |
| Neptune | grafo, serverless pequeño | 400 – 800 |
| SageMaker | 2 instancias de endpoint + entrenamiento diario | 500 – 900 |
| API Gateway + Lambda | API de recomendaciones | 100 – 250 |
| MSK Serverless o Kinesis | eventos de recomendaciones | 300 – 800 |
| EMR | jobs Spark diarios | 300 – 700 |
| MWAA | entorno pequeño | 350 – 450 |
| CloudWatch + SNS | métricas, logs, alertas | 50 – 150 |
| **Total estimado** | | **2,390 – 4,770** |
""",

'aws/patterns/recomendaciones-tiempo-real.md': """---
type: Pattern
title: Patrón de recomendaciones en tiempo real
description: Arquitectura de referencia inspirada en el caso público de Peloton en AWS.
tags: [ml, recomendaciones, referencia]
resource: https://www.youtube.com/watch?v=ym_Gz_zH7w8
---
# Patrón: recomendaciones en tiempo real

Referencia pública: Peloton, serie "This Is My Architecture" de AWS.

## Ruta batch (diaria)
1. Historial de miembros en S3.
2. Preprocesamiento con Spark en EMR; los mismos jobs pueblan el feature store y el grafo.
3. Entrenamiento (en el caso de referencia, en EKS con GPUs) y registro del modelo.
4. Orquestación con Airflow en MWAA.

## Ruta en línea (cada vez que el miembro inicia una sesión)
1. El dispositivo llama a un servicio que pide las recomendaciones.
2. Un orquestador obtiene features (DynamoDB), candidatos (grafo) y el modelo vigente.
3. Un servicio de inferencia puntúa los candidatos y devuelve la lista ordenada.
4. Lo recomendado se registra como evento (MSK) y llega a S3: ciclo de retroalimentación para reentrenar.

## Variante para equipos pequeños
- Sin experiencia en Kubernetes: reemplazar EKS por entrenamiento y endpoints de SageMaker.
- MSK Serverless o Kinesis Data Streams en lugar de un clúster MSK dimensionado a mano.
- Servicios de consultoría que lo implementan: [plataforma](../../company/services/realtime-recs-platform.md)
  y [pipeline MLOps](../../company/services/mlops-pipeline.md).
""",

}

for path, content in OKF_FILES.items():
    dest = OKF / path
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(content, encoding="utf-8")

print(f"{len(OKF_FILES)} OKF files written to {OKF.resolve()}")
!find okf -name '*.md' | sort

In [ ]:
import frontmatter

def validate_okf():
    invalid = [str(p) for p in OKF.rglob("*.md")
               if p.name != "log.md" and not frontmatter.load(p).get("type")]
    print("✅ Valid OKF" if not invalid else f"❌ Missing 'type': {invalid}")

validate_okf()

In [ ]:
@tool
def read_okf(path: str = "index.md") -> str:
    """Reads the NubeAndina OKF knowledge base (in Spanish): services, rates, AWS fundamentals and reference costs.
    Start at 'index.md' and follow the links. Examples: 'company/rate-card.md', 'aws/basics/pricing-models.md'."""
    p = (OKF / path).resolve()
    if not p.is_relative_to(OKF.resolve()):
        return "Path outside OKF."
    if p.is_dir():
        p = p / "index.md"
    if not p.exists():
        return f"{path} does not exist. Read the folder's index.md."
    post = frontmatter.load(p)
    return f"[okf:{p.relative_to(OKF.resolve())}] {post.metadata}\n\n{post.content}"

# ✅ Checkpoint
print(read_okf.invoke({"path": "company/rate-card.md"}))

### El agente de propuestas
Mismo `create_agent` del Paso 4 de `raglab.ipynb`, con herramientas nuevas (`search_transcripts`, `read_okf`), un procedimiento explícito en `SYSTEM` y memoria por sesión (`InMemorySaver`). Sin búsqueda web: todo sale de la llamada y del OKF. El prompt está en inglés, pero le pide al agente responder siempre en español. `ask()` envía un mensaje al agente dentro de una sesión (`thread_id`).

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM = """You are the technical sales proposal generator for NubeAndina Consulting.
The sales team uses you to prepare AWS services proposals based on calls with clients.
Always answer in Spanish: the clients, the call and the knowledge base are in Spanish.
Process:
1. search_transcripts: the client's needs, budget, timelines, team and constraints.
2. read_okf starting at index.md: services in company/services/, prices in company/rate-card.md,
   pattern in aws/patterns/, fundamentals in aws/basics/, AWS costs in aws/basics/pricing-models.md.
Rules:
- Consulting prices ONLY from company/rate-card.md. If a service is not there, say so and do not quote it.
- AWS costs ONLY from aws/basics/pricing-models.md, labeled as "estimado ilustrativo".
- Respect the client's constraints (team, budget, timeline). If a fact is missing, list it under Supuestos.
Proposal format (section titles in Spanish): 1) Necesidad del cliente (cite [client Xs-Ys])
2) Arquitectura propuesta 3) Servicios AWS y costo mensual estimado
4) Servicios de consultoría (table with price) 5) Total y supuestos
6) Fuentes (okf: paths and call fragments).
For specific questions, answer briefly, with the source."""

agent = create_agent(
    model=create_llm(),
    tools=[search_transcripts, read_okf],
    system_prompt=SYSTEM,
    checkpointer=InMemorySaver(),
)

def ask(msg, session="demo"):
    r = agent.invoke({"messages": [{"role": "user", "content": msg}]},
                     config={"configurable": {"thread_id": session},
                             "recursion_limit": 40})    # caps tool-call loops (~20 tool calls)
    return r

# The sales team writes in Spanish
r = ask("Genera la propuesta para RitmoFit: plataforma de recomendaciones en tiempo real.")
proposal = r["messages"][-1].text   # .text joins the text blocks; .content is a list (reasoning + text)
from IPython.display import Markdown, display
display(Markdown(proposal))

### Servidor Flask + chat
Mismo patrón de `raglab.ipynb`. Abrimos el puerto con el proxy de Colab (no requiere localtunnel). `/ask_bot` responde con `ask()` y `/upload_audio` recibe la consulta del botón **📤 Subir consulta**: transcribe el audio (o carga el `.json`) y lo agrega a `transcript_db` sin reiniciar. Volver a ejecutar la celda detiene el servidor anterior y libera el puerto.

In [ ]:
import json, threading
from flask import Flask, request, jsonify, render_template_string
from werkzeug.serving import make_server
from werkzeug.utils import secure_filename
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

PORT = 5002

# Re-running this cell: stop the previous server and free the port
if "server" in globals():
    server.shutdown()
    server.server_close()
    print("♻️ Previous server stopped.")

app = Flask(__name__)

@app.route("/")
def home():
    return render_template_string(html)

@app.route("/ask_bot", methods=["POST"])
def ask_bot():
    data = request.form or request.json
    r = ask(data["msg"], data.get("session_id", "default"))
    return jsonify({"msg": r["messages"][-1].text})

@app.route("/upload_audio", methods=["POST"])
def upload_audio():
    global transcript_db
    f = request.files["audio"]
    client = request.form.get("client", "").strip() or "unknown"
    path = f"/content/{secure_filename(f.filename)}"
    f.save(path)
    segs = json.load(open(path)) if path.endswith(".json") else transcribe(path)   # .json = backup transcript
    if not segs:
        return jsonify({"error": "empty transcript"}), 400
    new_docs = segments_to_docs(segs, client)
    if "transcript_db" in globals():
        transcript_db.add_documents(new_docs)
    else:                                 # no call indexed yet: create the index here
        transcript_db = FAISS.from_documents(new_docs, OpenAIEmbeddings(model=EMBEDDINGS_MODEL))
    return jsonify({"client": client, "chunks": len(new_docs), "minutes": round(segs[-1]["end"] / 60, 1)})

# Bind here (not inside the thread) so a busy port raises an error in this cell
server = make_server("0.0.0.0", PORT, app, threaded=True)
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
print(f"✅ Server running on port {PORT}")

from google.colab.output import eval_js
print("Open the chat:", eval_js(f"google.colab.kernel.proxyPort({PORT})"))
# Fallback if the proxy doesn't open: !npx localtunnel --port 5002  (as in raglab.ipynb)

In [ ]:
# Test uploading a new call without restarting (requires another MP3 in /content)
# !curl -F audio=@/content/call2.mp3 -F client=other http://localhost:5002/upload_audio

In [ ]:
# Stop the server and free the port
server.shutdown()
server.server_close()
print("✅ Server stopped.")

## Etapa 3 · Jev: evaluaciones System 1

**Objetivo:** medir cada respuesta con decisiones **tipadas** y rápidas (70–500 ms), no con ensayos.

| Tipo | Devuelve | Lo usamos para (clave en `QUESTIONS`) |
| --- | --- | --- |
| `choice` | una etiqueta + `confidence` | ¿Qué tipo de respuesta dio el agente? (`type`) |
| `score` | posición en una escala ordenada | ¿Qué tan completa es? (`coverage`) |
| `noul` | probabilidad sí/no calibrada (0–1) | ¿Precios del rate card? ¿Respeta restricciones? (`prices_ok`, `constraints_ok`) |

System 1 (rápido, barato, tipado) vs System 2 (jueces que razonan, como G-Eval: lentos y caros). Docs: [jevtypesafeai.com/docs](https://jevtypesafeai.com/docs).

In [ ]:
import os, requests

JEV_URL = "https://jevtypesafeai.com/api/v1/decide"

def jev(state, questions, model="jev-latest"):
    api_key = os.environ.get("JEV_API_KEY")       # set in the setup cell (02)
    if not api_key:
        raise RuntimeError("Missing JEV_API_KEY in Colab secrets")
    r = requests.post(JEV_URL, headers={"Authorization": f"Bearer {api_key}"},
                      json={"model": model, "state": state, "questions": questions}, timeout=15)
    r.raise_for_status()
    return r.json()

# Minimal test
jev("The client asks for a migration to Azure", {"is_aws": {"type": "noul", "instructions": "Is the request about AWS?"}})

In [ ]:
# Golden set: inputs stay in Spanish, as the sales team would write them
golden = [
  {"input": "Genera la propuesta para RitmoFit: plataforma de recomendaciones en tiempo real.",
   "type": "proposal"},
  {"input": "¿Cuánto cuesta un Well-Architected Review?", "type": "direct_answer"},
  {"input": "¿Cuál es el presupuesto de RitmoFit para la fase 1?", "type": "direct_answer"},
  {"input": "¿Pueden operar la plataforma cada mes después del MVP?", "type": "direct_answer"},
  {"input": "Cotiza una migración a Azure para RitmoFit", "type": "out_of_catalog"},
]

QUESTIONS = {
  "type": {"type": "choice",
    "instructions": "What type of answer did the agent give?",
    "criteria": {
      "proposal": "proposal with architecture, services and prices",
      "direct_answer": "answers a specific question",
      "out_of_catalog": "states that the service is not offered and gives no price",
      "hallucination": "offers a service or price that is not in the context"}},
  "prices_ok": {"type": "noul",
    "instructions": "Does every consulting price in the answer appear exactly the same in the rate-card.md context? If there are no prices, answer yes."},
  "constraints_ok": {"type": "noul",
    "instructions": "Does the answer respect the client's constraints in the transcript: managed services without Kubernetes, a budget of USD 25,000 and an MVP in 10 weeks?"},
  "coverage": {"type": "score",
    "instructions": "How complete is the answer relative to what the user asked?",
    "criteria": ["does not answer", "answers part of it", "answers the main point, missing details", "complete and with sources"]},
}

In [ ]:
import pandas as pd, time
from langchain_core.messages import ToolMessage

CTX_CHARS = 4000        # cap each tool result sent to Jev: the full OKF reads make the request heavy

rows = []
for i, c in enumerate(golden):
    print(f"[{i + 1}/{len(golden)}] {c['input'][:70]}", flush=True)
    row = {"case": i}
    try:
        t0 = time.time()
        r = ask(c["input"], session=f"eval-{time.time_ns()}")       # new session: no memory between cases
        row["agent_s"] = round(time.time() - t0, 1)
        state = {"question": c["input"],
                 "tool_context": [str(m.content)[:CTX_CHARS] for m in r["messages"] if isinstance(m, ToolMessage)],
                 "agent_answer": r["messages"][-1].text}
        t0 = time.time()
        a = jev(state, QUESTIONS)
        ans = a["answers"]
        row.update({"type": ans["type"]["choice"], "type_ok": ans["type"]["choice"] == c["type"],
                    "prices_ok": round(ans["prices_ok"]["noul"], 2),
                    "constraints_ok": round(ans["constraints_ok"]["noul"], 2),
                    "coverage": ans["coverage"]["score"],
                    "jev_ms": int((time.time() - t0) * 1000), "jev_usd": a["usage"]["cost_usd"]})
        print(f"    ✅ agent {row['agent_s']} s · jev {row['jev_ms']} ms", flush=True)
    except Exception as e:                              # one failing case doesn't stop the run
        row["error"] = f"{type(e).__name__}: {e}"[:200]
        print(f"    ⚠️ {row['error']}", flush=True)
    rows.append(row)

df = pd.DataFrame(rows)
df

**Regla de aprobación (acordarla en clase):** `type_ok`, `prices_ok` ≥ 0.8, `coverage` ≥ 2. `constraints_ok` aplica al caso 0.

**Itera:** cambia una sola cosa (quita la regla "Consulting prices ONLY from company/rate-card.md" del `SYSTEM`, o el `MODEL`), vuelve a crear el agente, re-ejecuta y compara las tablas.

**Reto:** la arquitectura de referencia (Peloton) usa EKS, pero RitmoFit no sabe Kubernetes. ¿Lo detecta `constraints_ok`?

In [ ]:
# Extra: inline guardrail. Jev checks every answer before it reaches the sales team.
from langchain_core.messages import ToolMessage

PRICES_MIN = 0.5          # below this, the answer is blocked
CONSTRAINTS_MIN = 0.5     # below this, a proposal is shown with a warning

def evaluate(question, r):
    """Runs the Jev QUESTIONS on an agent result; returns {type, prices_ok, constraints_ok, coverage}."""
    ctx = [str(m.content)[:4000] for m in r["messages"] if isinstance(m, ToolMessage)]
    a = jev({"question": question, "tool_context": ctx, "agent_answer": r["messages"][-1].text}, QUESTIONS)
    ans = a["answers"]
    return {"type": ans["type"]["choice"],
            "prices_ok": round(ans["prices_ok"]["noul"], 2),
            "constraints_ok": round(ans["constraints_ok"]["noul"], 2),
            "coverage": ans["coverage"]["score"]}

def ask_bot_with_guardrail(msg, session):
    """Agent + Jev. Returns (answer, checks); checks is None if Jev is unavailable."""
    r = ask(msg, session)
    answer = r["messages"][-1].text
    try:
        checks = evaluate(msg, r)
    except Exception as e:
        print(f"   🛡️ Jev unavailable, answer not checked: {e}")
        return answer, None
    checks["blocked"] = checks["type"] == "hallucination" or checks["prices_ok"] < PRICES_MIN
    # Messages shown to the sales team, so they stay in Spanish
    if checks["blocked"]:
        answer = "⚠️ La propuesta no pasó la revisión de precios: revísala contra el rate card antes de enviarla."
    elif checks["type"] == "proposal" and checks["constraints_ok"] < CONSTRAINTS_MIN:
        answer = ("> ⚠️ **Revisar antes de enviar:** la propuesta podría no respetar las restricciones del cliente.\n\n"
                  + answer)
    return answer, checks

## 🎁 Bonus · Arquitectura con iconos AWS y diapositiva con precios

**Objetivo:** convertir la propuesta en un diagrama y una diapositiva.

1. El LLM **extrae** de la propuesta un JSON tipado (`Architecture`: componentes, conexiones, costos).
2. La librería `diagrams` lo dibuja con los **iconos oficiales de AWS** (determinista, gratis) en `draw()`.
3. Los totales se calculan en **código** (`totals`), nunca en el modelo de imagen.
4. **`gpt-image-2.5-flare`** (`IMAGE_MODEL`) de OpenAI recibe el diagrama + los precios y devuelve una diapositiva 16:9. Usa la misma `OPENAI_API_KEY`; calidad `medium` para la clase.

Las etiquetas del diagrama y el texto de la diapositiva quedan en español porque son para el cliente.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

Service = Literal["user", "device", "api_gateway", "lambda", "sagemaker", "dynamodb", "neptune",
                  "msk", "kinesis", "s3", "emr", "mwaa", "cloudwatch", "sns", "eks"]

class Component(BaseModel):
    id: str = Field(description="short identifier, no spaces")
    service: Service
    label: str = Field(description="short name shown under the icon, in Spanish")
    group: Literal["online", "data", "batch", "observability"]

class Connection(BaseModel):
    source: str
    target: str
    label: str = ""

class CostItem(BaseModel):
    item: str = Field(description="cost line as written in the proposal, in Spanish")
    usd: float = Field(description="for ranges use the high value")
    kind: Literal["consulting", "aws_monthly"]

class Architecture(BaseModel):
    title: str = Field(description="diagram title, in Spanish")
    components: list[Component]
    connections: list[Connection]
    costs: list[CostItem]

extractor = create_llm().with_structured_output(Architecture)

def extract_architecture(proposal):
    return extractor.invoke(
        "Extract the AWS architecture and the costs from this proposal. Use only services and figures present in the text. "
        "Keep the title, labels and cost items in Spanish.\n\n"
        + proposal)

if "proposal" not in globals():       # normally created by the agent cell (12)
    print("No proposal yet: asking the agent for one…")
    proposal = ask("Genera la propuesta para RitmoFit: plataforma de recomendaciones en tiempo real.")["messages"][-1].text

arch = extract_architecture(proposal)
arch.model_dump()

In [ ]:
from diagrams import Diagram, Cluster, Edge
from diagrams.aws.analytics import EMR, ManagedStreamingForKafka, KinesisDataStreams
from diagrams.aws.compute import Lambda, EKS
from diagrams.aws.database import Dynamodb, Neptune
from diagrams.aws.integration import SNS
from diagrams.aws.management import Cloudwatch, AmazonManagedWorkflowsApacheAirflow
from diagrams.aws.ml import Sagemaker
from diagrams.aws.network import APIGateway
from diagrams.aws.storage import S3
from diagrams.onprem.client import User, Client
from IPython.display import Image, display

ICONS = {"user": User, "device": Client, "api_gateway": APIGateway, "lambda": Lambda,
         "sagemaker": Sagemaker, "dynamodb": Dynamodb, "neptune": Neptune, "msk": ManagedStreamingForKafka,
         "kinesis": KinesisDataStreams, "s3": S3, "emr": EMR, "mwaa": AmazonManagedWorkflowsApacheAirflow,
         "cloudwatch": Cloudwatch, "sns": SNS, "eks": EKS}
# Cluster titles are shown to the client, so they stay in Spanish
GROUPS = {"online": "En línea", "data": "Datos", "batch": "Batch diario", "observability": "Observabilidad"}

def draw(arch, filename="architecture"):
    with Diagram(arch.title, filename=filename, outformat="png", show=False, direction="LR",
                 graph_attr={"fontsize": "20", "pad": "0.4"}):
        nodes = {}
        for g, name in GROUPS.items():
            comps = [c for c in arch.components if c.group == g]
            if comps:
                with Cluster(name):
                    for c in comps:
                        nodes[c.id] = ICONS[c.service](c.label)
        for k in arch.connections:
            if k.source in nodes and k.target in nodes:
                nodes[k.source] >> Edge(label=k.label) >> nodes[k.target]
    return f"{filename}.png"

png = draw(arch)
display(Image(png))

In [ ]:
def cost_totals(arch):
    """Totals per kind, computed in code (never by the image model)."""
    return {kind: sum(c.usd for c in arch.costs if c.kind == kind) for kind in ("consulting", "aws_monthly")}

totals = cost_totals(arch)
print(f"Consulting (one-time): USD {totals['consulting']:,.0f}")
print(f"AWS (monthly, illustrative estimate): USD {totals['aws_monthly']:,.0f}")
pd.DataFrame([c.model_dump() for c in arch.costs])

In [ ]:
import base64
from openai import OpenAI

openai_client = OpenAI()

def make_slide(arch, png, filename="proposal_slide.png", quality="medium"):   # "low" to save money in class
    """Turns the diagram into a 16:9 proposal slide with the pricing table. Returns the PNG path."""
    totals = cost_totals(arch)
    lines = "\n".join(f"- {c.item}: USD {c.usd:,.0f}" + (" / mes" if c.kind == "aws_monthly" else "")
                      for c in arch.costs)
    prompt = f"""Turn this AWS architecture diagram into a sales proposal slide, 16:9 format,
clean and professional style, white background. All text on the slide must be in Spanish.
Keep the same AWS icons, services and arrows from the diagram. Do not add or remove services.
On the right, add a pricing table with exactly these lines:
{lines}
Total consultoría (una vez): USD {totals['consulting']:,.0f}
Total AWS estimado ilustrativo: USD {totals['aws_monthly']:,.0f} / mes
Title: {arch.title}. Provider: NubeAndina Consulting."""
    with open(png, "rb") as diagram:
        res = openai_client.images.edit(       # edits from the diagram: keeps icons and arrows
            model=IMAGE_MODEL,
            image=diagram,
            prompt=prompt,
            size="1536x1024",                  # landscape, slide-like
            quality=quality,
        )
    with open(filename, "wb") as f:
        f.write(base64.b64decode(res.data[0].b64_json))
    return filename

slide = make_slide(arch, png)
display(Image(slide))
# Without an input diagram: inside make_slide use openai_client.images.generate(model=IMAGE_MODEL, prompt=prompt, size="1536x1024")

**Discusión:** compara las dos imágenes. El diagrama de `diagrams` es exacto pero sobrio; la del modelo de imagen es más vistosa pero puede alterar textos o cifras. ¿Cuál enviarías al cliente? ¿Cómo verificarías la diapositiva antes de enviarla?

## 🚀 Servidor final: chat + guardrails de Jev + diagrama y diapositiva

Mismo servidor de la Etapa 1, ahora reutilizando lo que construimos en el notebook:

- `/ask_bot` → `ask_bot_with_guardrail()`: el agente responde y Jev lo evalúa (`QUESTIONS`). Si `type = hallucination` o `prices_ok` < `PRICES_MIN`, la respuesta se bloquea. Si la propuesta tiene `constraints_ok` bajo, se muestra con una advertencia. El chat muestra el resultado de Jev bajo cada respuesta.
- `/visuals` → solo para propuestas aprobadas: `extract_architecture()` → `draw()` (iconos AWS) → `make_slide()` (`IMAGE_MODEL`). El chat muestra el diagrama y la diapositiva.
- `/upload_audio` → el botón **📤 Subir consulta**, igual que antes.

Requiere haber ejecutado las celdas anteriores, incluido el Bonus. Cada paso se imprime en la salida de esta celda.

In [ ]:
import json, threading, time, uuid
from pathlib import Path
from flask import Flask, request, jsonify, render_template_string, send_from_directory
from werkzeug.serving import make_server
from werkzeug.utils import secure_filename
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

PORT = 5002
IMG_DIR = Path("/content/chat_images")
IMG_DIR.mkdir(exist_ok=True)

# Re-running this cell: stop the previous server and free the port
if "server" in globals():
    server.shutdown()
    server.server_close()
    print("♻️ Previous server stopped.")

app = Flask(__name__)

@app.route("/")
def home():
    return render_template_string(html)

@app.route("/ask_bot", methods=["POST"])
def ask_bot():
    data = request.form or request.json
    print(f"💬 {data['msg']}", flush=True)
    t0 = time.time()
    answer, checks = ask_bot_with_guardrail(data["msg"], data.get("session_id", "default"))   # agent + Jev
    print(f"   🤖 {time.time() - t0:.1f} s · 🛡️ {checks}", flush=True)
    is_proposal = checks["type"] == "proposal" and not checks["blocked"] if checks else False
    return jsonify({"msg": answer, "checks": checks, "visuals": is_proposal})

@app.route("/visuals", methods=["POST"])
def visuals():
    name = uuid.uuid4().hex[:8]
    try:
        t0 = time.time()
        arch = extract_architecture(request.json["proposal"])
        png = draw(arch, filename=str(IMG_DIR / f"arch_{name}"))
        print(f"   📐 {png} ({time.time() - t0:.1f} s)", flush=True)
        slide = make_slide(arch, png, filename=str(IMG_DIR / f"slide_{name}.png"))
        print(f"   🖼️ {slide} ({time.time() - t0:.1f} s)", flush=True)
    except Exception as e:
        print(f"   ❌ visuals failed: {type(e).__name__}: {e}", flush=True)
        return jsonify({"error": str(e)}), 500
    return jsonify({"diagram": f"/img/{Path(png).name}", "slide": f"/img/{Path(slide).name}",
                    "totals": cost_totals(arch)})

@app.route("/img/<name>")
def img(name):
    return send_from_directory(IMG_DIR, name)

@app.route("/upload_audio", methods=["POST"])
def upload_audio():
    global transcript_db
    f = request.files["audio"]
    client = request.form.get("client", "").strip() or "unknown"
    path = f"/content/{secure_filename(f.filename)}"
    f.save(path)
    print(f"📥 {path} · client={client}", flush=True)
    segs = json.load(open(path)) if path.endswith(".json") else transcribe(path)   # .json = backup transcript
    if not segs:
        return jsonify({"error": "empty transcript"}), 400
    new_docs = segments_to_docs(segs, client)
    if "transcript_db" in globals():
        transcript_db.add_documents(new_docs)
    else:                                 # no call indexed yet: create the index here
        transcript_db = FAISS.from_documents(new_docs, OpenAIEmbeddings(model=EMBEDDINGS_MODEL))
    print(f"   📝 {len(new_docs)} chunks indexed", flush=True)
    return jsonify({"client": client, "chunks": len(new_docs), "minutes": round(segs[-1]["end"] / 60, 1)})

# Bind here (not inside the thread) so a busy port raises an error in this cell
server = make_server("0.0.0.0", PORT, app, threaded=True)
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
print(f"✅ Server running on port {PORT}")

from google.colab.output import eval_js
print("Open the chat:", eval_js(f"google.colab.kernel.proxyPort({PORT})"))